# Current debugging file for validating difference between ML and QA
- Validating outputted models work as intended
- Troubleshooting

In [ ]:
import onnx
import numpy as np
from onnx import helper, numpy_helper, TensorProto
import onnxruntime as ort
import Utils
import matplotlib.pyplot as plt
import importlib
import pandas as pd
import seaborn as sns
import Evaluation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
# INPUTFILE = [f"../ML-debug/AnalysisResults-ML/FwdMatchMLCandidates-{i}.root" for i in range(1, 21)]
# print(INPUTFILE)
INPUTFILE = "Data/OOLHC25i4testing.root" #"Data/sample4testing.root" #"PbPbLHC26b13_FIXED.root" # "OO-LHC25i4_FIXED.root"
MODELFILE = "lgbmpbpballfeaturespt03.onnx" #"Models_Proper/OO_WithAbsFewerFeatures.onnx"

In [ ]:
FEATURES = ['DeltaDirection', 'PullPt', 'APullPhi', 'PullTanl', 'PtMFT',
       'CPhiPhiMFT', 'DeltaR', 'SameSign', 'DeltaEta', 'DeltaTanl',
       'RelPtDiff', 'CYYMFT', 'C1Pt1PtMFT', 'PullPhi', 'CXXMFT',
       'C1PtPhiMFT', 'YMCH', 'XMCH', 'DeltaPt', 'ADeltaPhi', 'PullR',
       'PullX', 'PullY', 'CXYMFT', 'CTglTglMCH', 'DeltaPhi', 'C1PtXMFT'] # prior OO features EEE
# FEATURES = ['RelPtDiff', 'DeltaDirection', 'SameSign', 'DeltaR', 'PtMFT',
#        'ADeltaPhi', 'CXXMFT', 'CPhiPhiMCH', 'PullPt', 'CYYMFT',
#        'C1Pt1PtMFT', 'CPhiPhiMFT', 'CTglTglMCH', 'C1Pt1PtMCH', 'DeltaPt',
#        'TanlMFT', 'PullR', 'ADeltaX', 'DeltaTanl', 'PullTanl', 'ADeltaY',
#        'C1PtPhiMFT', 'CTglTglMFT', 'etaMFT', 'DeltaEta', 'APullPhi',
#        ] # PbPb V0 with few features

# FEATURES = ['RelPtDiff', 'SameSign', 'PtMFT', 'PullPt', 'CPhiPhiMFT',
#        'C1Pt1PtMFT', 'CTglTglMCH', 'CPhiPhiMCH', 'TanlMFT', 'CXXMFT',
#        'CYYMFT', 'DeltaDirection', 'DeltaR', 'ADeltaPhi', 'etaMFT',
#        'PullR', 'DeltaPt', 'C1PtPhiMFT', 'DeltaEta', 'ADeltaX',
#        'APullPhi', 'DeltaTanl', 'CYYMCH', 'ADeltaY', 'CTglTglMFT',
#        'PullTanl', 'CXXMCH', 'InvQPtMFT', 'PtMCH', 'C1Pt1PtMCH', 'PullY',
#        'CXYMFT', 'APullX', 'DeltaX', 'APullY', 'DeltaPhi', 'C1PtXMFT',
#        'DeltaY', 'CTglXMCH', 'etaMCH', 'PullX', 'YMCH', 'TanlMCH', 'XMCH',
#        'CPhiXMFT', 'CTglXMFT', 'CPhiYMFT', 'C1PtYMFT', 'CTglPhiMFT',
#        'PullPhi', 'XMFT', 'YMFT', 'CPhiYMCH', 'CTglYMFT', 'PhiMFT',
#        'PhiMCH', 'C1PtTglMCH', 'CTglYMCH', 'C1PtTglMFT', 'CPhiXMCH'] # PbPb V1 with more features
FEATURES = ['XMCH',
 'YMCH',
 'PhiMCH',
 'TanlMCH',
 'InvQPtMCH',
 'CXXMCH',
 'CYYMCH',
 'CPhiPhiMCH',
 'CTglTglMCH',
 'C1Pt1PtMCH',
 'CXYMCH',
 'CPhiYMCH',
 'CPhiXMCH',
 'CTglXMCH',
 'CTglYMCH',
 'CTglPhiMCH',
 'C1PtXMCH',
 'C1PtYMCH',
 'C1PtPhiMCH',
 'C1PtTglMCH',
 'XMFT',
 'YMFT',
 'PhiMFT',
 'TanlMFT',
 'InvQPtMFT',
 'TrackTypeMFT',
 'CXXMFT',
 'CYYMFT',
 'CPhiPhiMFT',
 'CTglTglMFT',
 'C1Pt1PtMFT',
 'CXYMFT',
 'CPhiYMFT',
 'CPhiXMFT',
 'CTglXMFT',
 'CTglYMFT',
 'CTglPhiMFT',
 'C1PtXMFT',
 'C1PtYMFT',
 'C1PtPhiMFT',
 'C1PtTglMFT',
 'etaMCH',
 'etaMFT',
 'DeltaEta',
 'DeltaX',
 'DeltaY',
 'DeltaPhi',
 'ADeltaPhi',
 'ADeltaX',
 'ADeltaY',
 'DeltaTanl',
 'DeltaR',
 'RMFT',
 'SameSign',
 'PtMCH',
 'PtMFT',
 'DeltaPt',
 'RelPtDiff',
 'PullPt',
 'PullX',
 'PullY',
 'PullR',
 'PullPhi',
 'PullTanl',
 'APullX',
 'APullY',
 'APullPhi',
 'DeltaDirection']

In [ ]:
len(FEATURES)

In [ ]:
# np.seterr(all='raise')

In [ ]:
df = Utils.get_dataframe(INPUTFILE, folder_name="DF_*")
df = Utils.process_dataframe(df)
# from hipe4ml.tree_handler import TreeHandler
# df = TreeHandler(INPUTFILE, "O2fwdmlcand", folder_name="DF_*").get_data_frame()

In [ ]:
# df = Utils.subsample(df, frac = 0.3)
df = df[df['PtMCH'] > 0.7]

In [ ]:
df_train, df_val, df = Evaluation.Splitter(df, val_frac=0.1, test_frac = 0.2)

In [ ]:
df[FEATURES].describe()

In [ ]:
sess = ort.InferenceSession(MODELFILE)
input_name = sess.get_inputs()[0].name

In [ ]:
print(input_name, sess.get_inputs()[0].shape, sess.get_inputs()[0].type)

In [ ]:
pred = sess.run(None, {input_name: df[FEATURES].to_numpy(dtype=np.float32)})

In [ ]:
print([(o.name, o.shape, o.type) for o in sess.get_outputs()])
print([(type(p), np.shape(p)) for p in pred])
print(pred[1][0])

In [ ]:
pred = sess.run(
    None,
    {input_name: df[FEATURES].to_numpy(dtype=np.float32)}
)

df['score'] = [p[1] for p in pred[1]]

In [ ]:
importlib.reload(Utils)
mch_cols = ["PtMCH", "etaMCH"] # MCH cols we can actually use for these metrics as they require the underlying group structure to be preserved
common_cols= mch_cols + ['MFTMult']
#TODO: confirm these preserve grouping. Add non mch group preserving structures... like possibly MFTMult if it remains defined based on the best chi2 tracks around a mch track - correspond t
for entry in common_cols:   
    Utils.plot_metrics_vs_feature(df=df,feature=entry, threshold = 0.8, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.0, trim_high=0.1, Nsigma=1.0)

In [ ]:
for output in sess.get_outputs():
    print(output.name, output.shape, output.type)

In [ ]:
df.head(5)

In [ ]:
df.describe()

In [ ]:
# #Hack append some high score wrong matches to force the bins to work as in the qa-task
# df_new = df.copy()
# for i in range(0,4): 
#     # Copy a single row as a DataFrame
#     df_testing = df[df["MatchLabel"] == i].iloc[[1]].copy()

#     # Modify the score column
#     df_testing["score"] = 1.0

#     # Append back
#     df_new = pd.concat([df_new, df_testing], ignore_index=True)

In [ ]:
match_groups = Utils.build_match_groups(df)

Utils.draw_feature("score", match_groups=match_groups, density=False, log= True)

In [ ]:
# df_leader = df.loc[df.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
# match_groups_leader = Utils.build_match_groups(df_leader)
# Utils.draw_all_features(features=["score"], match_groups=match_groups_leader, density=False, per=0.0)

In [ ]:
# print(df[FEATURES+["score"]].head(100).to_string())  

In [ ]:
# df[FEATURES+["score"]].head(100).to_csv("onnx_validation_output.csv", index=True)

In [ ]:
df_subsample = df[["score"]].head(1000)

In [ ]:
bins = np.linspace(0.01,1.0,100).round(2)
n_true = [match_groups['True']["score"].between(x-0.01,x).sum() for x in bins]
n_wrong = [match_groups['Wrong']["score"].between(x-0.01,x).sum() for x in bins]
n_fake= [match_groups['Fake']["score"].between(x-0.01,x).sum() for x in bins]
n_decay= [match_groups['Decay']["score"].between(x-0.01,x).sum() for x in bins]
print(bins)

In [ ]:
df_histo_example = pd.DataFrame({
    'bin' : bins,
    'True' : n_true,
    'Wrong' : n_wrong,
    'Decay' : n_decay,
    'Fake' : n_fake
})


In [ ]:
sns.barplot(x=df_histo_example['bin'], y = df_histo_example['True'])
sns.barplot(x=df_histo_example['bin'], y = df_histo_example['Wrong'])
sns.barplot(x=df_histo_example['bin'], y = df_histo_example['Decay'])
sns.barplot(x=df_histo_example['bin'], y = df_histo_example['Fake'])
plt.show()

In [ ]:
df_histo_example.to_csv("histovalidation.csv", index=False)

In [ ]:
fig, ax = plt.subplots()
counts, bins, patches = ax.hist(df['MatchLabel'], bins=10, edgecolor='white',log=False)

# 2. Automatically add text labels on top of the bars
ax.bar_label(patches, padding=3)
plt.show()

In [ ]:
nums = df['MatchLabel'].value_counts()

In [ ]:
nums.sum()

In [ ]:
df_histo_example['Decay'].sum()